# 05 — Qini Gain and Policy Evaluation

This is the key notebook.

The objective is to evaluate whether the uplift model ranks customers in a useful order for campaign targeting.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np

from uplift_marketing.data import generate_synthetic_uplift_data

from sklearn.model_selection import train_test_split

from uplift_marketing.features import infer_feature_spec, split_features_treatment_outcome
from uplift_marketing.models import TLearner
from uplift_marketing.uplift import attach_uplift_scores, make_uplift_deciles, uplift_by_group
from uplift_marketing.evaluation import qini_curve, area_under_qini, policy_value_at_fraction
from uplift_marketing.policy import assign_campaign_segments, campaign_action_table


data = generate_synthetic_uplift_data(n_rows=5_000, random_state=42)
spec = infer_feature_spec(data, extra_excluded=["customer_id", "visit"])
x, treatment, y = split_features_treatment_outcome(data, spec)

x_train, x_test, t_train, t_test, y_train, y_test, raw_train, raw_test = train_test_split(
    x, treatment, y, data, test_size=0.3, random_state=42, stratify=treatment.astype(str) + "_" + y.astype(str)
)

model = TLearner(spec=spec, random_state=42).fit(x_train, t_train, y_train)
p_control, p_treated = model.predict_potential_outcomes(x_test)
scored = attach_uplift_scores(raw_test.reset_index(drop=True), p_control=p_control, p_treated=p_treated)
scored["uplift_decile"] = make_uplift_deciles(scored)
scored["campaign_segment"] = assign_campaign_segments(scored)

uplift_by_group(scored, group_col="uplift_decile")


In [ ]:
curve = qini_curve(
    outcome=scored["conversion"].to_numpy(),
    treatment=scored["treatment"].to_numpy(),
    uplift_score=scored["uplift_score"].to_numpy(),
)
area_under_qini(curve)


In [ ]:
policy_value_at_fraction(scored, fraction=0.2)


## Campaign action table

The model should not end with a score. It should support an action table.


In [ ]:
campaign_action_table(scored)
